In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import requests
import os
import time
import codecs
from tqdm import tqdm
import matplotlib.pyplot as plt
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from collections import Counter
import json
import requests
from dotenv import load_dotenv

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline
from huggingface_hub import login

/Users/jing/Documents/RaShips/accelerators/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def read_crunchbase():

    companies = pd.read_excel("/Users/jing/Documents/RaShips/web_scraping/WAYBACK/firmlevelall(n=1439).xlsx")       

    return companies

def clean_domain_url(website):
        s =  re.sub(r"www\.","",website)
        s =  re.sub(r"home\.","",s)
        s  = re.sub(r"\:.*","",s)
        s = re.sub(r"http(s?)://(\w+)\.(\w+)\.(\w+)",r"\3.\4",s)
        return s

def split_wayback_url(wayback_url):
    original_url = re.sub(r'http://web.archive.org/web/\d+/',"",wayback_url)
    website_piece = re.sub(r"http(s?)\://","", original_url)

    try:
        (domain, address) = website_piece.split("/", 1)            
    except ValueError:
        domain  = website_piece
        address = ""

    domain = clean_domain_url(domain)
    
    return (domain, address)

def store_page(wayback_url, html, output_folder, crawled_year, year_folder = True):
    (domain, address) = split_wayback_url(wayback_url)

    if year_folder:
        base_directory = "{0}/{1}/{2}".format(output_folder, domain, crawled_year)
    else:
        base_directory = output_folder + "/" + domain

        
    if not os.path.exists(base_directory):
            os.makedirs(base_directory)

    if address == "":
        address = "homepage.html"

    address = address.replace("/","_")
    if len(address) > 255:
        address = address[:255]

    file_path = base_directory +  "/" + address
    outfile = codecs.open(file_path, "w",'utf-8')
    outfile.write(html)
    outfile.close()
    
def clean_downloaded_data(data):
    
    data['timestamp'] = data.apply(lambda x: x['complete_url'].split('/')[4], axis = 1)
    #data['len_timestammp'] = data.apply(lambda x: len(x['timestamp']), axis = 1)
    data['timestamp'] = data.apply(lambda x: re.search(r'/web/(\d{14})/', x['complete_url']).group(1), axis = 1)

    data['download_date'] = data.apply(lambda x: dt.strptime(x['timestamp'][:8], '%Y%m%d'), axis = 1)
    data['download_time'] = data.apply(lambda x: dt.strptime(x['timestamp'][8:], '%H%M%S'), axis = 1)
    data['download_year'] = data.apply(lambda x: x['download_date'].year, axis = 1)

    #data['domain'] = data['url'].apply(split_wayback_url)
    #data['website'] = 'www.' + data['domain'] 

    return data

In [4]:
complete_data = pd.read_excel('accelerators_complete_data_with_responses.xlsx')
complete_data.head()

,accelerator_id,website,accelerator_name,seeddb_cohort_id,cohort_name,location,acceleration_date,duration,funding,equity,...,timestamp,download_date,download_time,download_year,snap_distance,llama3_funding,llama3_duration,llama3_equity,llama3_mentorship,llama3_demo_days
0,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c01,10-xcelerator 06/01/11,"Columbus, OH US",2011-06-01,NaN,NaN,NaN,...,20110613102235,2011-06-13,1900-01-01 10:22:35,2011,12,"Up to $100,000",Q2 (Spring 2012),0%,NaN,NaN
1,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c02,10-xcelerator 01/01/12,"Columbus, OH US",2012-01-01,NaN,NaN,NaN,...,20120107013515,2012-01-07,1900-01-01 01:35:15,2012,6,"Up to $100,000",The acceleration period mentioned in the text ...,0%,NaN,NaN
2,a02,http://betaspring.com,Betaspring,a02c01,Betaspring Summer 2009,"Providence, RI, US",2009-07-01,NaN,NaN,NaN,...,20101224050550,2010-12-24,1900-01-01 05:05:50,2010,541,0,Winter,0%,"According to the text, the accelerator program...",Post Demo Day Press Weekly Reader - Week Four
3,a02,http://betaspring.com,Betaspring,a02c02,Betaspring Summer 2010,"Providence, RI, US",2010-07-01,NaN,NaN,NaN,...,20101224050550,2010-12-24,1900-01-01 05:05:50,2010,176,0,Winter,0%,"According to the text, here's what I found:\n\...",Post Demo Day Press
4,a02,http://betaspring.com,Betaspring,a02c03,Betaspring Summer 2011,"Providence, RI, US",2011-07-01,NaN,NaN,NaN,...,20110707230422,2011-07-07,1900-01-01 23:04:22,2011,6,0,June 14th,0%,"According to the text:\n\n""What is Betaspring?...",NaN


# First Attempt

In [53]:
i = 45
trial_url = complete_data.complete_url.values[i]
year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
domain, address = split_wayback_url(trial_url)
text = []

with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.txt', 'r') as file:
    for line in file.readlines():
        text.append(line)

text

['Y Combinator\n',
 'The Wayback Machine - https://www.ycombinator.com/\n',
 'About\n',
 'FAQ\n',
 'Resources\n',
 'Apply for\n',
 'S2015\n',
 'batch.\n',
 'Apply\n',
 'About\n',
 'Slideshow of founders and startups at Y Combinator.\n',
 'Y Combinator created a new model for funding early stage\xa0startups.\n',
 'Twice a year we invest a small amount of money (\n',
 '$120k\n',
 ') in a large number of startups (most recently\xa085).\n',
 'The startups move to Silicon Valley for 3 months, during which we work intensively with them to get the company into the best possible shape and refine their pitch to investors. Each cycle culminates in Demo Day, when the startups present their companies to a carefully selected, invite-only\xa0audience.\n',
 'But YC doesn’t end on Demo Day. We and the YC alumni network continue to help founders for the life of their company, and\xa0beyond.\n',
 'Learn More\n',
 'Startups\n',
 'Since\n',
 '2005\n',
 ", we've funded over\n",
 '800\n',
 'startups.\n',
 '

In [58]:
with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
    html_content = file.read()
    soup = BeautifulSoup(html_content, "html.parser")

all_text = soup.get_text()
print(all_text)

  







Y Combinator                  Deadline for W15 was Oct 14.Apply Late        About    Slideshow of founders and startups at Y Combinator.   Y Combinator created a new model for funding early stage startups. Twice a year we invest a small amount of money ($120k) in a large number of startups (most recently 85). The startups move to Silicon Valley for 3 months, during which we work intensively with them to get the company into the best possible shape and refine their pitch to investors. Each cycle culminates in Demo Day, when the startups present their companies to a carefully selected, invite-only audience. But YC doesn't end on Demo Day. We and the YC alumni network continue to help founders for the life of their company, and beyond. Learn More      Startups         Since 2005, we've funded over 700 startups.   Y Combinator is a community of over 1,400 founders.   Our companies have a combined valuation of over $30B.        Quotes      Watch Incubated's Interview with YC Partn

In [59]:
all_text = all_text.split('\n')
filtered_text = []
for text in all_text:
    if text != '' and text != ' ':
        filtered_text.append(text)
filtered_text

['  ',
 'Y Combinator                  Deadline for W15 was Oct 14.Apply Late        About    Slideshow of founders and startups at Y Combinator.   Y Combinator created a new model for funding early stage startups. Twice a year we invest a small amount of money ($120k) in a large number of startups (most recently 85). The startups move to Silicon Valley for 3 months, during which we work intensively with them to get the company into the best possible shape and refine their pitch to investors. Each cycle culminates in Demo Day, when the startups present their companies to a carefully selected, invite-only audience. But YC doesn\'t end on Demo Day. We and the YC alumni network continue to help founders for the life of their company, and beyond. Learn More      Startups         Since 2005, we\'ve funded over 700\xa0startups.   Y Combinator is a community of\xa0over 1,400\xa0founders.   Our companies have a combined valuation of\xa0over\xa0$30B.        Quotes      Watch Incubated\'s Interv

In [ ]:
llm = 'google/flan-t5-base'
model = AutoModelForSeq2SeqLM.from_pretrained(llm)
tokenizer = AutoTokenizer.from_pretrained(llm)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
responses = []

for sentence in tqdm(filtered_text):

    #prompt = f'Is there any reference to funding to startups in the following sentence? Please answer only with "Yes" or "No": {sentence}'
    prompt = f'Can you please identify in the following text the amount of funding given to or raised by the startups?: {sentence}'

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=20,     # Limit output length
        do_sample=True,         # Enable randomness
        temperature=0.7,        # Controls creativity
        top_p=0.9,              # Nucleus sampling
        repetition_penalty=1.1  # Reduce repetition
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    responses.append(response)

100%|██████████| 88/88 [00:13<00:00,  6.61it/s]


In [35]:
responses

['No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No']

In [18]:
filtered_text = np.array(filtered_text)
responses = np.array(responses)

filtered_text[responses == 'Yes']

array(['Chicago Startups: Incubating and Accelerating | Excelerate Labs',
       'Press', 'About', 'Excelerate Labs', 'Details', 'Investors',
       'Apply!', 'Top 3 RankedAcceleratorProgram', 'High-Profile Demo',
       'Subscribe to Newsletter', '2. THE PROGRAM', 'Month 2',
       'FinanceCrash Course& Demo Day Prep',
       'Excelerate founders and their companies get access to Excelerate resources and relationships for life.',
       'Showcase to 500 investors at theHouse of Blues in Chicago',
       'performance results',
       '“Excelerate Labs is the catalyst for building a great company. Both the content of the program and its intensity are boot camp for what it takes to be a successful entrepreneur. And as if that wasn’t enough, they pair this training with access to an unbelievable network of mentors, advisors, venture capitalists, and fellow entrepreneurs. Without Excelerate, Food Genius would likely still be a passion project that we worked on in our spare time”',
       '

In [66]:
prompt = f'Can you please identify in the following text the amount of funding given to the startups?: {' '.join(filtered_text[:len(filtered_text)//2])}. Answer with either the amount of funding, or "No" if there is no amount of funding in the text.'

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=20,     # Limit output length
    do_sample=True,         # Enable randomness
    temperature=0.7,        # Controls creativity
    top_p=0.9,              # Nucleus sampling
    repetition_penalty=1.1  # Reduce repetition
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

In [67]:
response

'No'

In [105]:
prompt = f'Can you please identify in the following text the amount of funding given to the startups?: {' '.join(filtered_text[len(filtered_text)//2:])}'

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=20,     # Limit output length
    do_sample=True,         # Enable randomness
    temperature=0.5,        # Controls creativity
    top_p=0.9,              # Nucleus sampling
    repetition_penalty=1.1  # Reduce repetition
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


$120k


In [80]:
len(inputs['input_ids'][0])

456

# Google

In [106]:
llm = 'google/flan-t5-base'
model = AutoModelForSeq2SeqLM.from_pretrained(llm)
tokenizer = AutoTokenizer.from_pretrained(llm)

i = 45
trial_url = complete_data.complete_url.values[i]
year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
domain, address = split_wayback_url(trial_url)

with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
    html_content = file.read()
    soup = BeautifulSoup(html_content, "html.parser")

all_text = soup.get_text()

all_text = all_text.split('\n')
filtered_text = []
for text in all_text:
    if text != '' and text != ' ':
        filtered_text.append(text)

filtered_text = ' '.join(filtered_text)

prompt = f'Can you please identify in the following text the amount of funding given to the startups?: {filtered_text}'

inputs = tokenizer(prompt, return_tensors="pt")

responses = []

if len(inputs['input_ids'][0]) > 512:

    filtered_text_1 = filtered_text[:len(filtered_text)//2]
    filtered_text_2 = filtered_text[len(filtered_text)//2:]

    inputs_1 = tokenizer(prompt, return_tensors="pt")
    inputs_2 = tokenizer(prompt, return_tensors="pt")

    responses = []

    for i in range(10):

        outputs = model.generate(
            **inputs_1,
            max_new_tokens=20,     # Limit output length
            do_sample=True,         # Enable randomness
            temperature=0.5,        # Controls creativity
            top_p=0.9,              # Nucleus sampling
            repetition_penalty=1.1  # Reduce repetition
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        responses.append(response)

    for i in range(10):

        outputs = model.generate(
            **inputs_2,
            max_new_tokens=20,     # Limit output length
            do_sample=True,         # Enable randomness
            temperature=0.5,        # Controls creativity
            top_p=0.9,              # Nucleus sampling
            repetition_penalty=1.1  # Reduce repetition
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        responses.append(response)

else:

    for i in range(10):

        outputs = model.generate(
            **inputs,
            max_new_tokens=20,     # Limit output length
            do_sample=True,         # Enable randomness
            temperature=0.5,        # Controls creativity
            top_p=0.9,              # Nucleus sampling
            repetition_penalty=1.1  # Reduce repetition
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        responses.append(response)

In [120]:
def extract_responses(complete_data, i):

    trial_url = complete_data.complete_url.values[i]
    year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
    domain, address = split_wayback_url(trial_url)

    responses = []

    if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

        with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")

        all_text = soup.get_text()

        all_text = all_text.split('\n')
        filtered_text = []
        for text in all_text:
            if text != '' and text != ' ':
                filtered_text.append(text)

        filtered_text = ' '.join(filtered_text)

        prompt = f'Can you please identify in the following text the amount of funding given to the startups?: {filtered_text}'

        inputs = tokenizer(prompt, return_tensors="pt")

        if len(inputs['input_ids'][0]) > 512:

            filtered_text_1 = filtered_text[:len(filtered_text)//2]
            filtered_text_2 = filtered_text[len(filtered_text)//2:]

            prompt_1 = f'Can you please identify in the following text the amount of funding given to the startups?: {filtered_text_1}'
            prompt_2 = f'Can you please identify in the following text the amount of funding given to the startups?: {filtered_text_2}'

            inputs_1 = tokenizer(prompt_1, return_tensors="pt")
            inputs_2 = tokenizer(prompt_2, return_tensors="pt")

            responses = []

            for i in range(10):

                outputs = model.generate(
                **inputs_1,
                max_new_tokens=20,     # Limit output length
                do_sample=True,         # Enable randomness
                temperature=0.5,        # Controls creativity
                top_p=0.9,              # Nucleus sampling
                repetition_penalty=1.1  # Reduce repetition
            )

                response = tokenizer.decode(outputs[0], skip_special_tokens=True)

                responses.append(response)

            for i in range(10):

                outputs = model.generate(
                    **inputs_2,
                    max_new_tokens=20,     # Limit output length
                    do_sample=True,         # Enable randomness
                    temperature=0.5,        # Controls creativity
                    top_p=0.9,              # Nucleus sampling
                    repetition_penalty=1.1  # Reduce repetition
                )

                response = tokenizer.decode(outputs[0], skip_special_tokens=True)

                responses.append(response)

        else:

            for i in range(10):

                outputs = model.generate(
                    **inputs,
                    max_new_tokens=20,     # Limit output length
                    do_sample=True,         # Enable randomness
                    temperature=0.5,        # Controls creativity
                    top_p=0.9,              # Nucleus sampling
                    repetition_penalty=1.1  # Reduce repetition
                )

                response = tokenizer.decode(outputs[0], skip_special_tokens=True)

                responses.append(response)

    return responses

In [121]:
all_responses = []

for i in tqdm(range(complete_data.shape[0])):

    all_responses.append(extract_responses(complete_data=complete_data, i = i))

100%|██████████| 344/344 [1:14:48<00:00, 13.05s/it]  


In [122]:
all_responses

[[],
 [],
 ['amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups'],
 ['amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups'],
 ['amount of funding given to the startups',
  'amount of funding given to the startups'

In [124]:
with open('google_responses.json', 'w') as f:
    json.dump(all_responses, f)

In [137]:
with open('google_responses.json', 'r') as file:
    google_responses = json.load(file)
google_responses

[[],
 [],
 ['amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups'],
 ['amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups',
  'amount of funding given to the startups'],
 ['amount of funding given to the startups',
  'amount of funding given to the startups'

In [142]:
Counter(google_responses[10])

Counter({'$15 million': 10,
         'How BabbaCo redefined itself – an interview with Jessica Kim April 9, 2012 More blog': 5,
         'The amount of funding given to the startups is not specified.': 1,
         '': 1,
         'amount of funding given to the startups': 1,
         '$.': 1,
         'No': 1})

In [147]:
final_responses = []

for responses in google_responses:

    if len(responses) == 0:

        final_responses.append(responses)
    else:

        if len(responses) == 10:

            maj_responses = []
            counter = Counter(responses)
            max_counter = 0

            for k, v in counter.items():
                if v >= max_counter:
                    if re.search(r'\d', k):
                        maj_responses.append(k)
                        max_counter = v

        elif len(responses) == 20:

            maj_responses = []
            counter_1 = Counter(responses[:10])
            counter_2 = Counter(responses[10:])
            max_counter = 0

            for k, v in counter_1.items():
                if v >= max_counter:
                    if re.search(r'\d', k):
                        maj_responses.append(k)
                        max_counter = v

            max_counter = 0
            for k, v in counter_2.items():
                if v >= max_counter:
                    if re.search(r'\d', k):
                        maj_responses.append(k)
                        max_counter = v

        final_responses.append(maj_responses)

In [148]:
final_responses

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['Up to $20,000 in seed funding and access to other angel and institutional investors',
  '$20,000'],
 [],
 ['$15 million',
  'How BabbaCo redefined itself – an interview with Jessica Kim April 9, 2012 More blog'],
 ['$45 million', 'HAXLR8R', 'HAXLR8R :: Hardware Accelerator'],
 ['$50,000', '$1.5 billion', 'HAXLR8R 2013', 'HAXLR8R'],
 ['$1.6 billion', '$50,000', '$1 million', 'HAXLR8R'],
 ['up to $50K', '$500,000', 'HAXLR8R'],
 ['$500,000'],
 ['The program grants up to $500,000 per year', '$500,000'],
 [],
 ['$1 million', '0', 'Nov 1/Nov 1/Nov 1/Nov 1'],
 ['$500,000'],
 ['$500,000'],
 ['$1.5 million', '$500,000', 'A total of $50,000.'],
 ['$100,000', '$500,000', '2012, Copyright UpTech LLC, All Rights Reserved'],
 ['0', '$2.5 million', '$500,000', '$1.5 million'],
 ['0', '$500,000', '$2.5 million', '$1.5 million'],
 [],
 [],
 [],
 ['Y Combinator YouOS "Most Innovative" of 2006: #7 | Reddit'],
 ['Y Combinator YouOS "Most Innovative" of 2006: #7 |

# Mistral Attempt

In [ ]:
llm = 'mistralai/Mistral-7B-Instruct-v0.2'

load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")

login(token)

model = AutoModelForCausalLM.from_pretrained(
    llm
    #cache_dir="/Users/jing/.mounty/Seagate Basic"
)

tokenizer = AutoTokenizer.from_pretrained(
    llm
    #cache_dir="/Users/jing/.mounty/Seagate Basic"
    )

Loading checkpoint shards: 100%|██████████| 3/3 [02:47<00:00, 55.79s/it]


In [10]:
i = 40
trial_url = complete_data.complete_url.values[i]
year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
domain, address = split_wayback_url(trial_url)


if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

    with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
        html_content = file.read()
        soup = BeautifulSoup(html_content, "html.parser")

    all_text = soup.get_text()

    all_text = all_text.split('\n')
    filtered_text = []
    for text in all_text:
        if text != '' and text != ' ':
            filtered_text.append(text)

    filtered_text = ' '.join(filtered_text)

print(filtered_text)

Y Combinator Forbes | Wired | Mixergy |  Inc | WSJ | GQ | ATD | Fool | Time | BW | NYT |  NW "The most prestigious program for budding digital entrepreneurs" The application deadline was March 29 but you can still apply late. In 2005, Y Combinator developed a new model of startup funding. Twice a year we invest a small amount of money ($14-20k + an $80k note) in a large number of startups (most recently 46).  The startups move to Silicon Valley for 3 months, during which we work intensively with them to get the company into the best possible shape and refine their pitch to investors.  Each cycle culminates in  Demo Day,  when the startups present to a large audience of investors. But YC doesn't end on Demo Day.  We and the YC alumni network continue to help founders for the life of their company, and beyond. Since 2005 we've funded over 500 startups, including  Loopt,  Reddit, Clustrix,  Wufoo,  Scribd,  Xobni,  Omgpop,  Weebly,  Songkick,  Disqus,  Dropbox, Twitch,  Heroku,  A Thinkin

In [6]:
prompt = f'Can you please identify in the following text the amount of funding given to or raised by the startups?: {filtered_text}'

start = time.time()

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=20,     # Limit output length
    do_sample=True,         # Enable randomness
    temperature=0.7,        # Controls creativity
    top_p=0.9,              # Nucleus sampling
    repetition_penalty=1.1  # Reduce repetition
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
end = time.time()

print(f'Total time: {(end-start)/60} minutes')

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Total time: 76.88373696804047 minutes


In [7]:
response

'Can you please identify in the following text the amount of funding given to or raised by the startups?: Y Combinator Forbes | Wired | Mixergy |  Inc | WSJ | GQ | ATD | Fool | Time | BW | NYT |  NW "The most prestigious program for budding digital entrepreneurs" The application deadline was March 29 but you can still apply late. In 2005, Y Combinator developed a new model of startup funding. Twice a year we invest a small amount of money ($14-20k + an $80k note) in a large number of startups (most recently 46).  The startups move to Silicon Valley for 3 months, during which we work intensively with them to get the company into the best possible shape and refine their pitch to investors.  Each cycle culminates in  Demo Day,  when the startups present to a large audience of investors. But YC doesn\'t end on Demo Day.  We and the YC alumni network continue to help founders for the life of their company, and beyond. Since 2005 we\'ve funded over 500 startups, including  Loopt,  Reddit, Cl

# Ollama

## Single Attempt

In [6]:
i = 8
trial_url = complete_data.complete_url.values[i]
year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
domain, address = split_wayback_url(trial_url)


if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

    with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
        html_content = file.read()
        soup = BeautifulSoup(html_content, "html.parser")

    all_text = soup.get_text()

    all_text = all_text.split('\n')
    filtered_text = []
    for text in all_text:
        if text != '' and text != ' ':
            filtered_text.append(text)

    filtered_text = ' '.join(filtered_text)

print(filtered_text)

Home Details About Companies Contact News Apply FAQ Mentors Library Blog Videos Schedule Get Involved Make the leap. Take your start-up to the next level. Whether from business plan to demo, prototype to market, or breaking into profitability, Excelerate’s immersive 10-week program connects talented founding teams with the mentors, customers, and investors they need to build the great companies of tomorrow. Start-up capital Up to $20,000 in seed funding and access to other angel and institutional investors Expertise Rigorous educational forums and a dedicated mentor Connections Access to other Exelerate founders, experienced entrepreneurs, corporate partners, universities, and legal and business services Thinking about taking the leap?  APPLY NOW  Latest NewsAn error has occurred; the feed is probably down. Try again later.Blog  Videos


In [7]:
#prompt = f'Can you please identify in the following text the amount of funding given to the startups by the accelerator? {filtered_text} Please provide only the amount of funding, or 0 if it is not specified in the text. Only reply with the amount of funding or 0, no additional information or words.'
#prompt = f'Provide the duration of the acceleration period in the following text: {filtered_text}. Only reply with the duration, or with "None" if the information is not provided, no other words or text.'
#prompt = f'This is the website of an accelerator for startups. Identidy in the following text the equity of the startups which is taken by the accelerator. The equity needs to be a percentage between 0% and 15%. If there is no equity mentioned, reply with 0%. This is the text: {filtered_text}. Reply only with the percentage, with no other words or text.'

In [19]:
prompt = f'''
This is the website of an accelerator for startups. Accelerators can keep a share of the startups shares
as a profit, and this share is defined as equity. Identify in the following text the equity of the startups which is 
taken by the accelerator. The equity needs to be a percentage between 0% and 15%. If there is no equity mentioned, 
reply with 0%. The equity is usually explicity mentioned, so if there is no percentage in the text, reply with 0%.
This is the text: 
{filtered_text}
Reply only with the percentage of equity or 0% if there is none, with no other words or text.
'''

In [20]:
response = requests.post(
    'http://localhost:11434/api/generate',
    json={
        'model': 'llama3:8b',
        'prompt': prompt,
        'temperature': 0.5,
        'max_length': 20,
        'stream':False
    }
)

In [21]:
response.json()['response']

'0%'

## Loop

In [6]:
all_fundings = []

for i in tqdm(range(complete_data.shape[0])):

    time.sleep(10)
    
    trial_url = complete_data.complete_url.values[i]
    year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
    domain, address = split_wayback_url(trial_url)


    if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

        with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")

        all_text = soup.get_text()

        all_text = all_text.split('\n')
        filtered_text = []
        for text in all_text:
            if text != '' and text != ' ':
                filtered_text.append(text)

        filtered_text = ' '.join(filtered_text)

        prompt = f'Can you please identify in the following text the amount of funding given to the startups by the accelerator? {filtered_text} Please provide only the amount of funding, or 0 if it is not specified in the text. Only reply with the amount of funding or 0, no additional information or words.'

        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'mistral:instruct',
                'prompt': prompt,
                'temperature': 0.5,
                'max_length': 20,
                'stream':False

            }
        )

        all_fundings.append(response.json()['response'])

    else:

        all_fundings.append('missing')

    if len(all_fundings) % 50 == 0:
        with open('mistral_responses_fundings.json', 'w') as f:
            json.dump(all_fundings, f)

with open('mistral_response_fundings.json', 'w') as f:
    json.dump(all_fundings, f)

100%|██████████| 344/344 [1:38:17<00:00, 17.15s/it]


In [7]:
all_durations = []

for i in tqdm(range(complete_data.shape[0])):

    time.sleep(10)
    
    trial_url = complete_data.complete_url.values[i]
    year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
    domain, address = split_wayback_url(trial_url)


    if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

        with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")

        all_text = soup.get_text()

        all_text = all_text.split('\n')
        filtered_text = []
        for text in all_text:
            if text != '' and text != ' ':
                filtered_text.append(text)

        filtered_text = ' '.join(filtered_text)

        prompt = f'Provide the duration of the acceleration period in the following text: {filtered_text}. Only reply with the duration, or with "None" if the information is not provided, no other words or text.'

        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'mistral:instruct',
                'prompt': prompt,
                'temperature': 0.5,
                'max_length': 20,
                'stream':False

            }
        )

        all_durations.append(response.json()['response'])
    
    else:

        all_durations.append('missing')

    if len(all_durations) % 50 == 0:
        with open('mistral_responses_durations.json', 'w') as f:
            json.dump(all_durations, f)

with open('mistral_response_durations.json', 'w') as f:
    json.dump(all_durations, f)

  0%|          | 0/344 [00:00<?, ?it/s]

100%|██████████| 344/344 [1:47:01<00:00, 18.67s/it]  


In [8]:
all_equities = []

for i in tqdm(range(complete_data.shape[0])):

    time.sleep(10)
    
    trial_url = complete_data.complete_url.values[i]
    year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
    domain, address = split_wayback_url(trial_url)


    if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

        with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")

        all_text = soup.get_text()

        all_text = all_text.split('\n')
        filtered_text = []
        for text in all_text:
            if text != '' and text != ' ':
                filtered_text.append(text)

        filtered_text = ' '.join(filtered_text)

        prompt = f'''
                This is the website of an accelerator for startups. Accelerators can keep a share of the startups shares
                as a profit, and this share is defined as equity. Identify in the following text the equity of the startups which is 
                taken by the accelerator. The equity needs to be a percentage between 0% and 15%. If there is no equity mentioned, 
                reply with 0%. The equity is usually explicity mentioned, so if there is no percentage in the text, reply with 0%.
                This is the text: 
                {filtered_text}
                Reply only with the percentage of equity or 0% if there is none, with no other words or text.
                '''
        
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'mistral:instruct',
                'prompt': prompt,
                'temperature': 0.5,
                'max_length': 20,
                'stream':False

            }
        )

        all_equities.append(response.json()['response'])

    else:

        all_equities.append('missing')

    if len(all_equities) % 50 == 0:
        with open('mistral_responses_equities.json', 'w') as f:
            json.dump(all_equities, f)

with open('mistral_response_equities.json', 'w') as f:
    json.dump(all_equities, f)

  0%|          | 0/344 [00:00<?, ?it/s]

100%|██████████| 344/344 [1:36:37<00:00, 16.85s/it]


In [9]:
all_mentorships = []

for i in tqdm(range(complete_data.shape[0])):

    time.sleep(10)
    
    trial_url = complete_data.complete_url.values[i]
    year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
    domain, address = split_wayback_url(trial_url)


    if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

        with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")

        all_text = soup.get_text()

        all_text = all_text.split('\n')
        filtered_text = []
        for text in all_text:
            if text != '' and text != ' ':
                filtered_text.append(text)

        filtered_text = ' '.join(filtered_text)

        prompt = f'''
                This is the website of an accelerator for startups. Provide any information on the mentorship provided by mentors 
                or employees of the accelerator program. If there is no mention of mentorships, reply with "None". 
                This is the text: 
                {filtered_text}
                Reply only with the information about the mentorship or "None" if no information about mentorships is provided.
                '''
        
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'mistral:instruct',
                'prompt': prompt,
                'temperature': 0.5,
                'max_length': 200,
                'stream':False

            }
        )

        all_mentorships.append(response.json()['response'])
    
    else:

        all_mentorships.append('missing')

    if len(all_mentorships) % 50 == 0:
        with open('mistral_responses_mentorships.json', 'w') as f:
            json.dump(all_mentorships, f)

with open('mistral_responses_mentorships.json', 'w') as f:
    json.dump(all_mentorships, f)

100%|██████████| 344/344 [2:13:00<00:00, 23.20s/it]  


In [8]:
with open('mistral_responses_demo_days.json', 'r') as f:
    all_demo_days = json.load(f)

for i in tqdm(range(len(all_demo_days), complete_data.shape[0])):

    time.sleep(10)
    
    trial_url = complete_data.complete_url.values[i]
    year = pd.to_datetime(complete_data.acceleration_date.values[i]).year
    domain, address = split_wayback_url(trial_url)


    if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

        with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")

        all_text = soup.get_text()

        all_text = all_text.split('\n')
        filtered_text = []
        for text in all_text:
            if text != '' and text != ' ':
                filtered_text.append(text)

        filtered_text = ' '.join(filtered_text)

        prompt = f'''
                This is the website of an accelerator for startups. Demo days, which are events that take place at the end of the acceleration period and
                are organized to give start-ups the opportunity to pitch their business ideas in front of a
                crowd of interested people (e.g., business angels, venture capitalists, corporations,
                etc.). Provide any information on demo days in the follwing text. If there is no mention of demo days, reply with "None". 
                This is the text: 
                {filtered_text}
                Reply only with the information about the demo days or "None" if no information about demo days is provided.
                '''
        
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'mistral:instruct',
                'prompt': prompt,
                'temperature': 0.5,
                'max_length': 200,
                'stream':False

            }
        )

        all_demo_days.append(response.json()['response'])

    else:

        all_demo_days.append('missing')
        
    if len(all_demo_days) % 50 == 0:
        with open('mistral_responses_demo_days.json', 'w') as f:
            json.dump(all_demo_days, f)

with open('mistral_responses_demo_days.json', 'w') as f:
    json.dump(all_demo_days, f)

Exception ignored in: <function tqdm.__del__ at 0x116229ee0>
Traceback (most recent call last):
  File "/Users/jing/Documents/RaShips/accelerators/.venv/lib/python3.12/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Users/jing/Documents/RaShips/accelerators/.venv/lib/python3.12/site-packages/tqdm/std.py", line 1277, in close
    if self.last_print_t < self.start_t + self.delay:
       ^^^^^^^^^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'last_print_t'
100%|██████████| 44/44 [10:35<00:00, 14.45s/it]


In [9]:
with open('mistral_response_fundings.json', 'r') as f:
    mistral_fundings = json.load(f)

with open('mistral_response_durations.json', 'r') as f:
    mistral_durations = json.load(f)

with open('mistral_response_equities.json', 'r') as f:
    mistral_equities = json.load(f)

with open('mistral_responses_mentorships.json', 'r') as f:
    mistral_mentorships = json.load(f)

with open('mistral_responses_demo_days.json', 'r') as f:
    mistral_demo_days = json.load(f)

In [10]:
complete_data['mistral_funding'] = mistral_fundings
complete_data['mistral_duration'] = mistral_durations
complete_data['mistral_equity'] = mistral_equities
complete_data['mistral_mentorship'] = mistral_mentorships
complete_data['mistral_demo_days'] = mistral_demo_days
complete_data.head()

,accelerator_id,website,accelerator_name,seeddb_cohort_id,cohort_name,location,acceleration_date,duration,funding,equity,...,llama3_funding,llama3_duration,llama3_equity,llama3_mentorship,llama3_demo_days,mistral_funding,mistral_duration,mistral_equity,mistral_mentorship,mistral_demo_days
0,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c01,10-xcelerator 06/01/11,"Columbus, OH US",2011-06-01,NaN,NaN,NaN,...,"Up to $100,000",Q2 (Spring 2012),0%,NaN,NaN,missing,missing,missing,missing,missing
1,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c02,10-xcelerator 01/01/12,"Columbus, OH US",2012-01-01,NaN,NaN,NaN,...,"Up to $100,000",The acceleration period mentioned in the text ...,0%,NaN,NaN,missing,missing,missing,missing,missing
2,a02,http://betaspring.com,Betaspring,a02c01,Betaspring Summer 2009,"Providence, RI, US",2009-07-01,NaN,NaN,NaN,...,0,Winter,0%,"According to the text, the accelerator program...",Post Demo Day Press Weekly Reader - Week Four,0 (The text does not specify the amount of fun...,None (The text does not provide information a...,0% (No equity percentage is mentioned in the p...,Mentorship-driven program provided by Betaspr...,None (No mention of Demo Days in the provided...
3,a02,http://betaspring.com,Betaspring,a02c02,Betaspring Summer 2010,"Providence, RI, US",2010-07-01,NaN,NaN,NaN,...,0,Winter,0%,"According to the text, here's what I found:\n\...",Post Demo Day Press,0 (The text does not specify an amount for the...,The text does not provide a specific duration...,0%,Mentorship-driven program offered by Betaspri...,None (No mention of Demo Days in the provided...
4,a02,http://betaspring.com,Betaspring,a02c03,Betaspring Summer 2011,"Providence, RI, US",2011-07-01,NaN,NaN,NaN,...,0,June 14th,0%,"According to the text:\n\n""What is Betaspring?...",NaN,0 (The text does not specify an amount of fund...,The duration of the acceleration period is Ju...,0%,"Mentorship is provided by Betaspring, as it i...",None (No mention of demo days in the text pro...


In [11]:
complete_data.to_excel('accelerators_complete_data_with_responses.xlsx', index = False)

In [5]:
with open('llama3_response_fundings.json', 'r') as f:
    llama3_fundings = json.load(f)

with open('llama3_response_durations.json', 'r') as f:
    llama3_durations = json.load(f)

with open('llama3_response_equities.json', 'r') as f:
    llama3_equities = json.load(f)

with open('llama3_response_mentorships.json', 'r') as f:
    llama3_mentorships = json.load(f)

with open('llama3_response_demo_days.json', 'r') as f:
    llama3_demo_days = json.load(f)

In [6]:
complete_data['llama3_funding'] = llama3_fundings
complete_data['llama3_duration'] = llama3_durations
complete_data['llama3_equity'] = llama3_equities
complete_data['llama3_mentorship'] = llama3_mentorships
complete_data['llama3_demo_days'] = llama3_demo_days
complete_data.head()

,accelerator_id,website,accelerator_name,seeddb_cohort_id,cohort_name,location,acceleration_date,duration,funding,equity,...,timestamp,download_date,download_time,download_year,snap_distance,llama3_funding,llama3_duration,llama3_equity,llama3_mentorship,llama3_demo_days
0,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c01,10-xcelerator 06/01/11,"Columbus, OH US",2011-06-01,NaN,NaN,NaN,...,20110613102235,2011-06-13,1900-01-01 10:22:35,2011,12,"Up to $100,000",Q2 (Spring 2012),0%,None,None
1,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c02,10-xcelerator 01/01/12,"Columbus, OH US",2012-01-01,NaN,NaN,NaN,...,20120107013515,2012-01-07,1900-01-01 01:35:15,2012,6,"Up to $100,000",The acceleration period mentioned in the text ...,0%,None,None
2,a02,http://betaspring.com,Betaspring,a02c01,Betaspring Summer 2009,"Providence, RI, US",2009-07-01,NaN,NaN,NaN,...,20101224050550,2010-12-24,1900-01-01 05:05:50,2010,541,0,Winter,0%,"According to the text, the accelerator program...",Post Demo Day Press Weekly Reader - Week Four
3,a02,http://betaspring.com,Betaspring,a02c02,Betaspring Summer 2010,"Providence, RI, US",2010-07-01,NaN,NaN,NaN,...,20101224050550,2010-12-24,1900-01-01 05:05:50,2010,176,0,Winter,0%,"According to the text, here's what I found:\n\...",Post Demo Day Press
4,a02,http://betaspring.com,Betaspring,a02c03,Betaspring Summer 2011,"Providence, RI, US",2011-07-01,NaN,NaN,NaN,...,20110707230422,2011-07-07,1900-01-01 23:04:22,2011,6,0,June 14th,0%,"According to the text:\n\n""What is Betaspring?...",None


In [10]:
complete_data = pd.read_excel('accelerators_complete_data_with_responses.xlsx')
complete_data.head()

,accelerator_id,website,accelerator_name,seeddb_cohort_id,cohort_name,location,acceleration_date,duration,funding,equity,...,llama3_funding,llama3_duration,llama3_equity,llama3_mentorship,llama3_demo_days,mistral_funding,mistral_duration,mistral_equity,mistral_mentorship,mistral_demo_days
0,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c01,10-xcelerator 06/01/11,"Columbus, OH US",2011-06-01,NaN,NaN,NaN,...,"Up to $100,000",Q2 (Spring 2012),0%,NaN,NaN,missing,missing,missing,missing,missing
1,a01,http://fisher.osu.edu/centers/entrepreneurship...,10-xcelerator,a01c02,10-xcelerator 01/01/12,"Columbus, OH US",2012-01-01,NaN,NaN,NaN,...,"Up to $100,000",The acceleration period mentioned in the text ...,0%,NaN,NaN,missing,missing,missing,missing,missing
2,a02,http://betaspring.com,Betaspring,a02c01,Betaspring Summer 2009,"Providence, RI, US",2009-07-01,NaN,NaN,NaN,...,0,Winter,0%,"According to the text, the accelerator program...",Post Demo Day Press Weekly Reader - Week Four,0 (The text does not specify the amount of fun...,None (The text does not provide information a...,0% (No equity percentage is mentioned in the p...,Mentorship-driven program provided by Betaspr...,None (No mention of Demo Days in the provided...
3,a02,http://betaspring.com,Betaspring,a02c02,Betaspring Summer 2010,"Providence, RI, US",2010-07-01,NaN,NaN,NaN,...,0,Winter,0%,"According to the text, here's what I found:\n\...",Post Demo Day Press,0 (The text does not specify an amount for the...,The text does not provide a specific duration...,0%,Mentorship-driven program offered by Betaspri...,None (No mention of Demo Days in the provided...
4,a02,http://betaspring.com,Betaspring,a02c03,Betaspring Summer 2011,"Providence, RI, US",2011-07-01,NaN,NaN,NaN,...,0,June 14th,0%,"According to the text:\n\n""What is Betaspring?...",NaN,0 (The text does not specify an amount of fund...,The duration of the acceleration period is Ju...,0%,"Mentorship is provided by Betaspring, as it i...",None (No mention of demo days in the text pro...


In [ ]:
print((complete_data.llama3_funding.values == '0').mean()*100)
print((complete_data.llama3_funding.str.contains('million')).mean()*100)
print((complete_data.llama3_duration == 'None').mean()*100)
print((complete_data.llama3_equity == '0%').mean()*100)
print((complete_data.llama3_mentorship == 'None').mean()*100)
print((complete_data.llama3_demo_days == 'None').mean()*100)


38.372093023255815
2.3255813953488373
0.0
63.08139534883721
0.0
0.0


In [23]:
null_funding = complete_data.loc[complete_data.llama3_funding == '0', :]

i = 85
trial_url = null_funding.complete_url.values[i]
year = pd.to_datetime(null_funding.acceleration_date.values[i]).year
domain, address = split_wayback_url(trial_url)


if os.path.isfile(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html'):

    with open(f'/Users/jing/Documents/RaShips/accelerators/WAYBACK/2025/out/{domain}/{year}/homepage.html', "r", encoding="utf-8") as file:
        html_content = file.read()
        soup = BeautifulSoup(html_content, "html.parser")

    all_text = soup.get_text()

    all_text = all_text.split('\n')
    filtered_text = []
    for text in all_text:
        if text != '' and text != ' ':
            filtered_text.append(text)

    filtered_text = ' '.join(filtered_text)

filtered_text

'\tAlphaLab \xa0              Welcome to AlphaLab AlphaLab is a catalyst for launching the next generation of  software, game design and Internet-related companies. Created by Innovation Works, one of the nation’s most active seed-stage investors, AlphaLab provides  funding, free office space, expert business advisors and services through an  intensive six-month program in Pittsburgh. AlphaLab helps companies rapidly  develop their technology, gain user feedback from early alpha or beta releases  and move toward successful commercial launch. AlphaLab began its Summer/Fall program with its first six companies on June 2, 2008.  The next application cycle for AlphaLab will begin August 15, 2008. The application deadline for the Winter/Spring program will be October 15, 2008. contact us | innovation works | sitemap ©2008 Copyright Innovation Works   AlphaLab Homepage                 This site requires Flash 8.'

In [ ]:
#complete_data.to_excel('accelerators_complete_data_with_responses.xlsx', index = False)